# Lektion 01 - Einführung in KI-Agenten

Willkommen zur ersten Lektion im Kurs **KI-Agenten für Einsteiger**!

Ein **KI-Agent** ist ein Programm, das ein großes Sprachmodell (LLM) als seine Denkmaschine verwendet und *Aktionen* in der realen Welt ausführen kann — APIs aufrufen, Datenbanken abfragen oder Code ausführen — um im Auftrag eines Nutzers ein Ziel zu erreichen.

In diesem Notizbuch werden Sie Ihren ersten Agenten erstellen: einen **Reiseagenten**, der Urlaubsdestinationen empfiehlt. Dabei lernen Sie:

1. Wie man sich mit dem Azure AI Foundry Agent Service unter Verwendung des **Microsoft Agent Frameworks** verbindet.
2. Dem Agenten ein **Werkzeug** gibt — eine einfache Python-Funktion, die er aufrufen kann.
3. Den Agenten ausführt und seine Antwort überprüft.
4. Die Antwort des Agenten Token für Token streamt.


## Einrichtung

Bevor Sie dieses Notebook ausführen, stellen Sie sicher, dass Sie:

1. **Ein Azure AI Foundry-Projekt** mit einem bereitgestellten Chatmodell (z. B. `gpt-4o-mini`) haben.
2. **Mit der Azure CLI angemeldet sind** — führen Sie `az login` in Ihrem Terminal aus.
3. **Die erforderlichen Umgebungsvariablen gesetzt haben:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Ihr Azure AI Foundry-Projektendpunkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — der Name Ihres bereitgestellten Modells.

Die folgende Zelle installiert die benötigten Python-Pakete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Erstellen Ihres ersten Agents

Ein Agent benötigt zwei Dinge:

- **Anweisungen**, die ihm sagen, *wer er ist* und *wie er sich verhalten soll* (ein System-Prompt).
- **Werkzeuge** — Python-Funktionen, die mit `@tool` dekoriert sind und die der Agent aufrufen kann, um Informationen abzurufen oder Aktionen auszuführen.

Unten definieren wir ein einfaches Werkzeug, das eine Liste beliebter Urlaubsziele zurückgibt. Der Agent wird dieses Werkzeug verwenden, wenn ein Benutzer nach Reiseempfehlungen fragt.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming-Antworten

Für ein interaktiveres Erlebnis können Sie die Antwort des Agenten **streamen**. Anstatt auf die vollständige Antwort zu warten, liefert der Agent Textabschnitte, sobald sie generiert werden. Dies ist besonders nützlich in Chat-Oberflächen, in denen Sie die Ausgabe in Echtzeit anzeigen möchten.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man:

- **Einen Provider erstellt**, der über `AzureAIProjectAgentProvider` eine Verbindung zum Azure AI Foundry Agent Service herstellt.
- **Ein Tool definiert** mithilfe des `@tool` Dekorators, damit der Agent Ihre Python-Funktionen aufrufen kann.
- **Den Agenten ausführt** mit einer Benutzernachricht und dessen Antwort ausgibt.
- **Antworten streamt** für eine Echtzeitausgabe.

In der nächsten Lektion werden wir agentenbasierte Frameworks eingehender untersuchen und lernen, wie man Agenten leistungsfähigere Werkzeuge und mehrstufige Denkfähigkeiten gibt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mithilfe des KI-Übersetzungsdienstes [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, möchten wir darauf hinweisen, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
